In [2]:
import numpy as np 
import pandas as pd 

from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score

from plotly import express as px

from utils import plot_confusion_matrix, get_artifact_filename

import os

from json import loads

from joblib import load, dump

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

In [3]:
# Paths
BASE_DIR = '../'
PATH_TO_MODELS = os.path.join(BASE_DIR, "work/models")
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")
PATH_TO_IMAGES_DIR = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train_images")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")


In [4]:
study_lgb = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name="04 - LGB Multiclass CV",
                            load_if_exists = True)


lgb_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_lgb,'test')))

[I 2026-04-23 19:58:21,316] Using an existing study with name '04 - LGB Multiclass CV' instead of creating a new one.


In [5]:
lgb_dataset

,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed,pred
14696,1,Dione & Elora,1,307,307,2,1,0,0,2,...,2,0,41327,61b07b54adb97d4b5f3c2dec06a9943b,0,Dione and Elora are puppies of Rambo. Both are...,8f20e24ef,9.0,4,"[0.05403658978652445, 0.8983786538750579, 1.80..."
14823,1,Har-nee,24,103,307,2,1,2,4,2,...,1,0,41330,9cb2e5a10e24e0b09942013b8434c81f,0,We found Har-nee with a swollen and almost sev...,2d72ef0c4,2.0,4,"[0.08449944474673063, 0.8160906407310541, 0.86..."
2838,1,The Gorgeous 5 Beauties,2,307,0,2,2,7,0,2,...,5,0,41326,5c398b2e18b16f0db83c53e682eada42,0,Theses 5 very adorably cute white female puppi...,44cd12263,5.0,4,"[0.06513853187256904, 0.5542095066632708, 1.78..."
1848,2,Mochi,1,265,0,1,2,0,0,1,...,1,0,41401,6905e4fbe5658eef5f560b814898a5ee,2,Hello! My name is Mochi. I was rescued from a ...,210c4a637,6.0,2,"[0.1825669538150876, 1.3296144721187018, 1.719..."
669,2,Nala & Peach,9,266,266,2,2,4,6,2,...,2,0,41326,803457cd3660dda694086b51a11a5a39,0,Nala is a cat that's been born with 7 fingers ...,21493e6ea,8.0,4,"[0.10333243176485976, 0.6579241561765493, 1.61..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,2,Anak Nanya,8,266,0,3,1,2,0,2,...,3,50,41326,f14c2cfebbbafbc9ed1f500d082f3ec3,0,they r ol so cute :) it juz a matter me dun hv...,35f9818a7,14.0,4,"[0.053166958705348974, 0.5794076851073074, 1.1..."
12222,1,Poor Baby,3,307,0,1,5,0,0,2,...,1,0,41401,500c48db7b281eabec3c293160f4a71c,0,On behalf of Exotica Pets Healthy puppy availa...,46e25aa2b,2.0,1,"[0.056412161481926563, 1.3388499748497675, 1.5..."
10538,2,No Name,1,265,0,2,1,6,0,1,...,1,0,41401,ac9a633cf51a70f4a9842e6e1ba91fc9,0,sy jumpa kitten ni mengiau2 kat playground. ra...,d3692d2b2,2.0,1,"[0.19464504680747663, 1.9545304132273582, 1.36..."
11062,1,Pipi,1,307,0,2,1,5,7,2,...,6,0,41326,3ef66c1034bb6dc31314845457079483,0,"Health, cute and active puppies.",3c43b7541,1.0,4,"[0.0839823863050407, 0.7864658121553668, 1.835..."


In [6]:
MODEL_NAME = '04 ResNet'
MODEL_VERSION = '1.0.0'

study_resnet = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)

resnet_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_resnet,'test')))

[I 2026-04-23 19:58:28,178] Using an existing study with name '04 ResNet_1.0.0' instead of creating a new one.


In [7]:
resnet_dataset

,PetID,pred,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,...,Sterilized,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PhotoAmt,AdoptionSpeed


In [8]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(resnet_dataset[['PetID', 'pred']].rename({'pred':'resnet_pred_score'},axis=1),
                  on='PetID', how='outer')



merged_datasets['resnet_pred_score'] = [np.zeros(5) if type(i) is float else  i for i in merged_datasets['resnet_pred_score'] ]

In [9]:
merged_datasets['resnet_pred_score']

0       [0.0, 0.0, 0.0, 0.0, 0.0]
1       [0.0, 0.0, 0.0, 0.0, 0.0]
2       [0.0, 0.0, 0.0, 0.0, 0.0]
3       [0.0, 0.0, 0.0, 0.0, 0.0]
4       [0.0, 0.0, 0.0, 0.0, 0.0]
                  ...            
2994    [0.0, 0.0, 0.0, 0.0, 0.0]
2995    [0.0, 0.0, 0.0, 0.0, 0.0]
2996    [0.0, 0.0, 0.0, 0.0, 0.0]
2997    [0.0, 0.0, 0.0, 0.0, 0.0]
2998    [0.0, 0.0, 0.0, 0.0, 0.0]
Name: resnet_pred_score, Length: 2999, dtype: object

In [10]:
merged_datasets['blend_pred_score'] = [r['lgb_pred_score']+r['resnet_pred_score'] for i,r in merged_datasets.iterrows()]

In [11]:
merged_datasets['lgb_pred'] = [r.argmax() for r in merged_datasets['lgb_pred_score']]
merged_datasets['resnet_pred'] = [r.argmax() for r in merged_datasets['resnet_pred_score']]
merged_datasets['blended_pred'] = [r.argmax() for r in merged_datasets['blend_pred_score']]

In [12]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['lgb_pred'], 
                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['lgb_pred'], 
                                                                    weights='quadratic')))

In [13]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['resnet_pred'], 
                    title = 'Resnet Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['resnet_pred'], 
                                                                    weights='quadratic')))



In [14]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['blended_pred'], 
                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['blended_pred'], 
                                                                    weights='quadratic')))
